## Workflow Agents with Google ADK

Building on the previous challenges, this notebook creates a **verified answer pipeline**
using ADK workflow agents:

- **Root Agent** — routes greetings vs. questions
- **Greeter Agent** — friendly greeting responses
- **Answer Pipeline** (`SequentialAgent`) — orchestrates the full answer workflow:
  1. **Research Phase** (`ParallelAgent`) — two search agents run concurrently
     - **Search Agent** — finds the direct answer
     - **Context Agent** — finds background and related information
  2. **Synthesize Agent** — combines parallel research into a draft answer
  3. **Refinement Loop** (`LoopAgent`, max 2 iterations) — iteratively improves the draft
     - **Critique Agent** — evaluates the draft and suggests improvements
     - **Refine Agent** — rewrites based on critique feedback

State management via `output_key` and `session.state` tracks data flow
between agents, plus a history log records each step for observability.

## 1. Setup & Dependencies

In [1]:
!pip install google-adk -q
!pip install google-genai -q

In [2]:
import os
from typing import Optional

import google.auth
from google.genai import types
from google import genai

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent, LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.tool_context import ToolContext

## 2. Configuration

In [3]:
project = google.auth.default()[1]

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = project
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

LOCATION = "us-central1"
GEMINI_MODEL = "gemini-2.0-flash"

# Vertex AI GenAI client (used by the custom search tool)
genai_client = genai.Client(vertexai=True, project=project, location=LOCATION)

print(f"Project: {project}")
print(f"Location: {LOCATION}")
print(f"Model:   {GEMINI_MODEL}")

Project: qwiklabs-gcp-00-2dc5a476dd1b
Location: us-central1
Model:   gemini-2.0-flash


## 3. Tools

### Search Tool
A custom wrapper around Gemini with Google Search grounding.
We wrap it in a regular function (instead of using the built-in `google_search`
grounding tool) so it can coexist with ADK's internal transfer tools.

### State Management Tools
Helper tools that let agents read from and append to shared session state,
providing a persistent history across the pipeline.

In [4]:
def search_the_web(query: str) -> str:
    """Search the web for information using Google Search.

    Use this tool to find up-to-date information about any topic.

    Args:
        query: The search query to look up.

    Returns:
        A text summary of the search results.
    """
    response = genai_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=query,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
        ),
    )
    return response.text


def set_state_value(
    tool_context: ToolContext, field: str, value: str
) -> dict:
    """Set a single value in session state, overwriting any existing value.

    Use this to store a draft answer, critique, or any single value
    that downstream agents need to read via template substitution.

    Args:
        field: The state key to set (e.g., 'current_answer').
        value: The text value to store.

    Returns:
        A status dict confirming success.
    """
    tool_context.state[field] = value
    return {"status": "success", "field": field}


def append_to_state(
    tool_context: ToolContext, field: str, value: str
) -> dict:
    """Append a value to a list stored in session state.

    Use this to keep a running log of observations, critiques,
    or refinements across iterations.

    Args:
        field: The state key to append to (e.g., 'critique_history').
        value: The text value to append.

    Returns:
        A status dict confirming success.
    """
    existing = tool_context.state.get(field, [])
    tool_context.state[field] = existing + [value]
    return {"status": "success", "entries": len(existing) + 1}


def read_from_state(
    tool_context: ToolContext, field: str
) -> dict:
    """Read a value from session state.

    Use this to retrieve previously stored information like
    search results, drafts, or critique history.

    Args:
        field: The state key to read from.

    Returns:
        A dict containing the stored value, or a message if the key is empty.
    """
    value = tool_context.state.get(field, None)
    if value is None:
        return {"status": "not_found", "field": field}
    return {"status": "success", "value": value}

## 4. Callback Functions

Logging callbacks carried forward from previous challenges,
adapted for the workflow pipeline.

In [5]:
def _get_latest_user_text(llm_request: LlmRequest) -> Optional[str]:
    """Extract the most recent user text from an LlmRequest."""
    if not llm_request.contents:
        return None
    for content in reversed(llm_request.contents):
        if content.role != "user" or not content.parts:
            continue
        if any(getattr(p, "function_response", None) for p in content.parts):
            continue
        for part in content.parts:
            if getattr(part, "text", None):
                return part.text
    return None


def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Log user prompts before they are sent to the model."""
    user_text = _get_latest_user_text(llm_request)
    if user_text:
        print(
            f"  [LOG] User prompt -> {callback_context.agent_name}: "
            f"{user_text[:120]}{'...' if user_text and len(user_text) > 120 else ''}"
        )
    return None


def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Log model responses after generation."""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if getattr(part, "text", None):
                preview = part.text[:150]
                print(
                    f"  [LOG] Response <- {callback_context.agent_name}: "
                    f"{preview}{'...' if len(part.text) > 150 else ''}"
                )
            elif getattr(part, "function_call", None):
                print(
                    f"  [LOG] Tool call <- {callback_context.agent_name}: "
                    f"{part.function_call.name}()"
                )
    return None

## 5. Agent Definitions

### Architecture Overview

```
root_agent (Agent — LLM router)
├── greeter_agent (LlmAgent)
└── answer_pipeline (SequentialAgent)
    ├── research_phase (ParallelAgent)
    │   ├── search_agent    → state["search_results"]
    │   └── context_agent   → state["background_context"]
    ├── synthesize_agent         → state["current_answer"]
    └── refinement_loop (LoopAgent, max_iterations=2)
        ├── critique_agent  → state["critique"]
        └── refine_agent    → state["current_answer"] (overwritten)
```

Data flows between agents via `output_key` -> `session.state` -> `{template}` references in instructions.

### 5a. Greeter Agent

In [6]:
greeter_agent = LlmAgent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description=(
        "Handles greetings and casual conversation. Use this agent "
        "when the user says hello, hi, thanks, goodbye, or engages "
        "in small talk rather than asking a factual question."
    ),
    instruction="""
You are a friendly, helpful greeter. Respond warmly to the user's
greeting or casual message. Keep it brief and cheerful. If they
seem to have a question, encourage them to ask it.
""",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

### 5b. Research Phase — Parallel Search Agents

Two agents search concurrently:
- **search_agent** — looks for the direct answer
- **context_agent** — finds background and related information

Each writes to a separate state key via `output_key`.

In [7]:
search_agent = LlmAgent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Searches for the direct answer to a user's question.",
    instruction="""
You are a research agent. Use the search_the_web tool to find a
direct, factual answer to the user's question.

Return ONLY the key facts and data points you found. Include
specific numbers, dates, names, and sources where possible.
Do not add opinions or commentary — just the raw findings.
""",
    tools=[search_the_web],
    output_key="search_results",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

context_agent = LlmAgent(
    name="context_agent",
    model=GEMINI_MODEL,
    description="Finds background context and related information.",
    instruction="""
You are a background research agent. Use the search_the_web tool
to find additional context, history, or related information that
would enrich the answer to the user's question.

Focus on:
- Historical context or background
- Related facts that add depth
- Why this topic matters or is interesting

Return ONLY the supplementary facts. Do not repeat the direct
answer — focus on what makes the answer richer.
""",
    tools=[search_the_web],
    output_key="background_context",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

research_phase = ParallelAgent(
    name="research_phase",
    sub_agents=[search_agent, context_agent],
    description="Runs search and context agents in parallel.",
)

### 5c. Synthesize Agent

Combines the parallel research results into a single draft answer.

In [8]:
synthesize_agent = LlmAgent(
    name="synthesize_agent",
    model=GEMINI_MODEL,
    description="Combines research into a draft answer.",
    instruction="""
You are a synthesis agent. Combine the following research into a
clear, well-structured draft answer.

## Direct Search Results
{search_results}

## Background Context
{background_context}

Write a comprehensive answer that:
1. Leads with the direct answer to the question
2. Adds relevant context and background
3. Cites sources where available
4. Is well-organized and easy to read

IMPORTANT: After writing your answer, you MUST use the set_state_value
tool to store your complete answer with field='current_answer'.
Also use the append_to_state tool to log this draft to the
'answer_history' field for tracking progress across refinements.
""",
    tools=[set_state_value, append_to_state],
    output_key="current_answer",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

### 5d. Refinement Loop — Critique & Refine

A `LoopAgent` that iteratively improves the draft answer:
1. **Critique Agent** reviews the current answer and suggests improvements
2. **Refine Agent** rewrites based on the critique

Runs for `max_iterations=2` rounds of improvement.

In [9]:
critique_agent = LlmAgent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Evaluates the current answer and suggests improvements.",
    instruction="""
You are a critical reviewer. Evaluate the following answer and
provide specific, actionable suggestions for improvement.

## Current Answer
{current_answer}

Evaluate against these criteria:
- **Accuracy**: Are the facts correct and well-sourced?
- **Completeness**: Does it fully answer the question?
- **Clarity**: Is it well-organized and easy to understand?
- **Conciseness**: Is there unnecessary filler or repetition?

Provide your critique as a numbered list of specific improvements.
Be constructive — explain what to change AND why.

IMPORTANT: After writing your critique, you MUST use the set_state_value
tool to store your complete critique with field='critique'.
Also use the append_to_state tool to log your critique to the
'critique_history' field.
""",
    tools=[set_state_value, append_to_state],
    output_key="critique",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Rewrites the answer based on critique feedback.",
    instruction="""
You are a refinement agent. Rewrite the answer based on the
critique feedback provided.

## Current Answer
{current_answer}

## Critique & Suggestions
{critique}

Rewrite the answer addressing ALL of the critique points.
Maintain the good parts while improving the weak areas.
The result should be a polished, final-quality response.

IMPORTANT: After writing your refined answer, you MUST use the
set_state_value tool to store your complete refined answer with
field='current_answer'. Also use the append_to_state tool to log
this refined version to the 'answer_history' field.
""",
    tools=[set_state_value, append_to_state],
    output_key="current_answer",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critique_agent, refine_agent],
    max_iterations=2,
    description="Iteratively critiques and refines the answer.",
)

### 5e. Answer Pipeline & Root Agent

Wire everything together:
- `answer_pipeline` (`SequentialAgent`): research \u2192 synthesize \u2192 refine loop
- `root_agent` (`Agent`): LLM-based router that delegates to greeter or answer pipeline

In [10]:
answer_pipeline = SequentialAgent(
    name="answer_pipeline",
    description=(
        "Answers factual questions using a verified research pipeline. "
        "Use this for any question that requires looking up information, "
        "facts, history, science, current events, how-to questions, etc."
    ),
    sub_agents=[research_phase, synthesize_agent, refinement_loop],
)

root_agent = Agent(
    name="root_agent",
    model=GEMINI_MODEL,
    description="The main coordinating agent that routes user requests.",
    instruction="""
You are the main coordinating agent. Your job is to understand what
the user needs and delegate to the right specialist:

- **greeter_agent**: For greetings, casual conversation, thanks,
  or goodbyes.
- **answer_pipeline**: For ANY factual question, lookup, or request
  that requires searching for information.

Always delegate to a sub-agent. Do not try to answer questions
yourself. If the user's intent is unclear, ask a brief clarifying
question.
""",
    sub_agents=[greeter_agent, answer_pipeline],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

## 6. Test Harness

Run the root agent against prompts, printing events to show
sub-agent delegation, workflow progression, and state evolution.

In [11]:
async def test_workflow(prompts: list[str], show_state: bool = True) -> None:
    """Run the root agent against prompts, printing events and state."""
    session_service = InMemorySessionService()
    runner = Runner(
        agent=root_agent,
        app_name="workflow_agents",
        session_service=session_service,
    )

    for i, prompt in enumerate(prompts):
        session_id = f"test_session_{i}"

        await session_service.create_session(
            app_name="workflow_agents",
            user_id="test_user",
            session_id=session_id,
        )

        content = types.Content(
            role="user",
            parts=[types.Part(text=prompt)],
        )

        print(f"\n{'=' * 70}")
        print(f"User: {prompt}")
        print(f"{'=' * 70}")

        final_text = ""
        async for event in runner.run_async(
            user_id="test_user",
            session_id=session_id,
            new_message=content,
        ):
            agent_name = getattr(event, "author", "unknown")
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if getattr(part, "text", None):
                        print(f"  [{agent_name}] {part.text[:200]}{'...' if len(part.text) > 200 else ''}")
                    elif getattr(part, "function_call", None):
                        print(f"  [{agent_name}] -> {part.function_call.name}()")
                    elif getattr(part, "function_response", None):
                        print(f"  [{agent_name}] <- tool result received")

            if event.is_final_response() and event.content and event.content.parts:
                final_text = event.content.parts[0].text

        if final_text:
            print(f"\n  {'-' * 60}")
            print(f"  FINAL ANSWER:\n")
            print(f"  {final_text[:500]}{'...' if len(final_text) > 500 else ''}")

        # Show state evolution
        if show_state:
            session = await session_service.get_session(
                app_name="workflow_agents",
                user_id="test_user",
                session_id=session_id,
            )
            if session and session.state:
                print(f"\n  {'-' * 60}")
                print(f"  STATE KEYS: {list(session.state.keys())}")
                for key in ["answer_history", "critique_history"]:
                    entries = session.state.get(key, [])
                    if entries:
                        print(f"\n  {key}: {len(entries)} entries")
                        for j, entry in enumerate(entries):
                            preview = entry[:100] if isinstance(entry, str) else str(entry)[:100]
                            print(f"     [{j+1}] {preview}...")
        print()

### 6a. Greeting — Routed to `greeter_agent`

In [12]:
await test_workflow(["Hello! How are you today?"])


User: Hello! How are you today?
  [LOG] User prompt -> root_agent: Hello! How are you today?


  [LOG] Tool call <- root_agent: transfer_to_agent()
  [root_agent] -> transfer_to_agent()
  [root_agent] <- tool result received
  [LOG] User prompt -> greeter_agent: For context:
  [LOG] Response <- greeter_agent: Hi there! I'm doing well, thanks for asking. How can I help you today?

  [greeter_agent] Hi there! I'm doing well, thanks for asking. How can I help you today?


  ------------------------------------------------------------
  FINAL ANSWER:

  Hi there! I'm doing well, thanks for asking. How can I help you today?




### 6b. Factual Question — Full Pipeline

This triggers the complete workflow:
1. Parallel research (search + context)
2. Synthesis into draft
3. Two rounds of critique & refinement

In [13]:
await test_workflow(["Who won the most recent Super Bowl and what were the key highlights?"])


User: Who won the most recent Super Bowl and what were the key highlights?
  [LOG] User prompt -> root_agent: Who won the most recent Super Bowl and what were the key highlights?
  [LOG] Tool call <- root_agent: transfer_to_agent()
  [root_agent] -> transfer_to_agent()
  [root_agent] <- tool result received
  [LOG] User prompt -> search_agent: For context:
  [LOG] User prompt -> context_agent: For context:
  [LOG] Tool call <- search_agent: search_the_web()
  [search_agent] -> search_the_web()
  [LOG] Tool call <- context_agent: search_the_web()
  [search_agent] <- tool result received
  [context_agent] -> search_the_web()
  [LOG] User prompt -> search_agent: For context:
  [context_agent] <- tool result received
  [LOG] User prompt -> context_agent: For context:
  [LOG] Response <- search_agent: The Seattle Seahawks won Super Bowl LX in 2026, defeating the New England Patriots 29-13.

Key highlights:
* Uchenna Nwosu scored on a 44-yard interce...
  [search_agent] The Seattle Seahawks

### 6c. Another Question — Demonstrates Repeat Pipeline Execution

In [14]:
await test_workflow(["What is the James Webb Space Telescope and what has it discovered?"])


User: What is the James Webb Space Telescope and what has it discovered?
  [LOG] User prompt -> root_agent: What is the James Webb Space Telescope and what has it discovered?
  [LOG] Tool call <- root_agent: transfer_to_agent()
  [root_agent] -> transfer_to_agent()
  [root_agent] <- tool result received
  [LOG] User prompt -> search_agent: For context:
  [LOG] User prompt -> context_agent: For context:
  [LOG] Tool call <- search_agent: search_the_web()
  [search_agent] -> search_the_web()
  [LOG] Tool call <- context_agent: search_the_web()
  [LOG] Tool call <- context_agent: search_the_web()
  [search_agent] <- tool result received
  [context_agent] -> search_the_web()
  [context_agent] -> search_the_web()
  [LOG] User prompt -> search_agent: For context:
  [context_agent] <- tool result received
  [context_agent] <- tool result received
  [LOG] User prompt -> context_agent: For context:
  [LOG] Response <- context_agent: The James Webb Space Telescope (JWST) is designed to conduct 

### 6d. Mixed Prompts — Routing + Pipeline

In [15]:
await test_workflow(
    [
        "Hey there!",
        "What year did the Titanic sink and how many passengers survived?",
    ],
    show_state=True,
)


User: Hey there!
  [LOG] User prompt -> root_agent: Hey there!
  [LOG] Tool call <- root_agent: transfer_to_agent()
  [root_agent] -> transfer_to_agent()
  [root_agent] <- tool result received
  [LOG] User prompt -> greeter_agent: For context:
  [LOG] Response <- greeter_agent: Hello! How can I help you today?

  [greeter_agent] Hello! How can I help you today?


  ------------------------------------------------------------
  FINAL ANSWER:

  Hello! How can I help you today?



User: What year did the Titanic sink and how many passengers survived?
  [LOG] User prompt -> root_agent: What year did the Titanic sink and how many passengers survived?
  [LOG] Tool call <- root_agent: transfer_to_agent()
  [root_agent] -> transfer_to_agent()
  [root_agent] <- tool result received
  [LOG] User prompt -> search_agent: For context:
  [LOG] User prompt -> context_agent: For context:
  [LOG] Tool call <- context_agent: search_the_web()
  [context_agent] -> search_the_web()
  [LOG] Tool call <- s